In [37]:
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
base_path = "/content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package"

phase4_path = os.path.join(base_path, "outputs/phase4_sql_baseline")
phase5_path = os.path.join(base_path, "outputs/phase5_assistant_prototype")
phase6_path = os.path.join(base_path, "outputs/phase6_citation_guardrails")
phase7_path = os.path.join(base_path, "outputs/phase7_formal_evaluation")

os.makedirs(phase7_path, exist_ok=True)

db_path = os.path.join(phase4_path, "partnerlens.db")

print("Database path:", db_path)
print("Phase 7 output path:", phase7_path)

Database path: /content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/outputs/phase4_sql_baseline/partnerlens.db
Phase 7 output path: /content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/outputs/phase7_formal_evaluation


In [39]:
conn = sqlite3.connect(db_path)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables

,name
0,partner_master
1,partner_pricing
2,partner_metrics


In [40]:
def run_sql(query):
    return pd.read_sql_query(query, conn)

In [41]:
run_sql("SELECT * FROM partner_master LIMIT 5;")

,partner_id,partner_name,partner_type,industry_vertical,partner_size,partner_status,country,state,city,region,integration_type,sales_channel,risk_tier,kyc_status,pci_level,onboarding_date,relationship_manager_region,synthetic_record_flag,industry,status
0,P000001,NorthStar Commerce 0001,Technology Platform,Travel,SMB,Suspended,US,NY,New York,Northeast,API,Direct,High,Approved,Level 3,2019-06-15,Northeast,1,Travel,Suspended
1,P000002,Ironwood Market 0002,ISV,SaaS,Mid-Market,Pilot,US,VA,Norfolk,South,SDK,Direct,Low,Approved,Level 4,2020-02-13,South,1,SaaS,Pilot
2,P000003,Evergreen Commerce 0003,Technology Platform,Professional Services,SMB,Active,US,CO,Colorado Springs,West,API,Direct,Low,Approved,Level 3,2023-03-12,West,1,Professional Services,Active
3,P000004,Summit Platform 0004,ISO/Reseller,Gaming,SMB,Active,US,GA,Augusta,South,API,Referral,Medium,Approved,Level 4,2020-02-13,South,1,Gaming,Active
4,P000005,Prairie PayTech 0005,Merchant Acquirer,Healthcare,SMB,Active,US,CA,San Diego,West,API,Direct,High,Approved,Level 4,2022-03-09,West,1,Healthcare,Active


In [42]:
benchmark_questions = pd.read_csv(
    os.path.join(phase4_path, "phase4_benchmark_questions.csv")
)

baseline_sql_queries = pd.read_csv(
    os.path.join(phase4_path, "phase4_baseline_sql_queries.csv")
)

query_library_df = pd.read_csv(
    os.path.join(phase5_path, "phase5_query_library.csv")
)

print("Benchmark questions:", benchmark_questions.shape)
print("Baseline SQL queries:", baseline_sql_queries.shape)
print("Query library:", query_library_df.shape)

Benchmark questions: (10, 6)
Baseline SQL queries: (10, 2)
Query library: (10, 6)


In [43]:
benchmark_questions.head()

,question_id,business_question,expected_tables,expected_fields,answer_type,citation_required
0,Q001,Show me technology partners in Arizona with tr...,"partner_master, partner_metrics","partner_type, state, growth_pct",Filtered list,Yes
1,Q002,Which partners are on the Strategic pricing tier?,"partner_master, partner_pricing","partner_name, pricing_tier",Filtered list,Yes
2,Q003,Which partners have pricing reviews due in the...,"partner_master, partner_pricing","partner_name, review_due_date, contract_status",Filtered list,Yes
3,Q004,Which industry has the highest average transac...,"partner_master, partner_metrics","industry_vertical, transaction_volume",Ranking summary,Yes
4,Q005,Show the top 5 partners by revenue.,"partner_master, partner_metrics","partner_name, revenue",Top 5 ranking,Yes


In [44]:
baseline_sql_queries.head()

,question_id,sql_query
0,Q001,"\nSELECT\n pm.partner_id,\n pm.partner_n..."
1,Q002,"\nSELECT\n pm.partner_id,\n pm.partner_n..."
2,Q003,"\nSELECT\n pm.partner_id,\n pm.partner_n..."
3,Q004,"\nSELECT\n pm.industry_vertical,\n ROUND..."
4,Q005,"\nSELECT\n pm.partner_id,\n pm.partner_n..."


In [45]:
query_library_df.head()

,question_id,intent,example_question,keywords,sql,citations
0,Q001,technology_az_growth_above_20,Show me technology partners in Arizona with tr...,"technology, arizona, az, growth, 20",\n SELECT\n pm.partn...,"partner_master.partner_id, partner_master.part..."
1,Q002,strategic_pricing_partners,Which partners are on the Strategic pricing tier?,"strategic, pricing, tier, partners",\n SELECT\n pm.partn...,"partner_master.partner_id, partner_master.part..."
2,Q003,pricing_reviews_due_90_days,Which partners have pricing reviews due in the...,"pricing, review, due, 90, days",\n SELECT\n pm.partn...,"partner_master.partner_id, partner_master.part..."
3,Q004,highest_average_transaction_volume_by_industry,Which industry has the highest average transac...,"industry, highest, average, transaction, volume",\n SELECT\n pm.indus...,"partner_master.industry_vertical, partner_metr..."
4,Q005,top_5_partners_by_revenue,Show the top 5 partners by revenue.,"top, 5, partners, revenue",\n SELECT\n pm.partn...,"partner_master.partner_id, partner_master.part..."


In [46]:
query_library = {}

for _, row in query_library_df.iterrows():
    qid = row["question_id"]

    keywords = []
    if pd.notna(row["keywords"]):
        keywords = [k.strip() for k in str(row["keywords"]).split(",")]

    citations = []
    if pd.notna(row["citations"]):
        citations = [c.strip() for c in str(row["citations"]).split(",")]

    query_library[qid] = {
        "intent": row["intent"],
        "example_question": row["example_question"],
        "keywords": keywords,
        "sql": row["sql"],
        "citations": citations
    }

list(query_library.keys())

['Q001',
 'Q002',
 'Q003',
 'Q004',
 'Q005',
 'Q006',
 'Q007',
 'Q008',
 'Q009',
 'Q010']

In [47]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def match_question_to_intent(user_question, minimum_score=2):
    normalized_question = normalize_text(user_question)

    best_match = None
    best_score = 0

    for qid, item in query_library.items():
        score = 0

        for keyword in item["keywords"]:
            keyword_normalized = normalize_text(keyword)

            if keyword_normalized in normalized_question:
                score += 1

        if score > best_score:
            best_score = score
            best_match = qid

    if best_match is None or best_score < minimum_score:
        return None, best_score

    return best_match, best_score

In [48]:
match_question_to_intent("Which partners are on the Strategic pricing tier?")

('Q002', 4)

In [49]:
def validate_sql_safety(sql_query, allowed_tables=None):
    if allowed_tables is None:
        allowed_tables = ["partner_master", "partner_pricing", "partner_metrics"]

    sql_clean = str(sql_query).strip().lower()

    blocked_keywords = [
        "insert", "update", "delete", "drop", "alter", "truncate",
        "create", "replace", "attach", "detach", "pragma", "vacuum"
    ]

    for keyword in blocked_keywords:
        pattern = r"\b" + keyword + r"\b"
        if re.search(pattern, sql_clean):
            return False, f"Blocked SQL keyword detected: {keyword}"

    first_word = sql_clean.split()[0]

    if first_word not in ["select", "with"]:
        return False, "Only SELECT or WITH queries are allowed."

    # Identify CTE names such as: WITH latest_month AS (...)
    cte_names = re.findall(
        r"(?:with|,)\s+([a-zA-Z_][a-zA-Z0-9_]*)\s+as\s*\(",
        sql_clean
    )

    allowed_references = allowed_tables + cte_names

    referenced_tables = re.findall(
        r"\bfrom\s+([a-zA-Z_][a-zA-Z0-9_]*)|\bjoin\s+([a-zA-Z_][a-zA-Z0-9_]*)",
        sql_clean
    )

    flat_tables = []

    for table_pair in referenced_tables:
        for table in table_pair:
            if table:
                flat_tables.append(table)

    for table in flat_tables:
        if table not in allowed_references:
            return False, f"Unauthorized table referenced: {table}"

    return True, "SQL passed safety validation."

In [50]:
def get_database_schema(conn):
    schema = {}

    table_names = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    )["name"].tolist()

    for table in table_names:
        columns_df = pd.read_sql_query(f"PRAGMA table_info({table});", conn)
        schema[table] = columns_df["name"].tolist()

    return schema


database_schema = get_database_schema(conn)
database_schema

{'partner_master': ['partner_id',
  'partner_name',
  'partner_type',
  'industry_vertical',
  'partner_size',
  'partner_status',
  'country',
  'state',
  'city',
  'region',
  'integration_type',
  'sales_channel',
  'risk_tier',
  'kyc_status',
  'pci_level',
  'onboarding_date',
  'relationship_manager_region',
  'synthetic_record_flag',
  'industry',
  'status'],
 'partner_pricing': ['assignment_id',
  'partner_id',
  'pricing_plan_id',
  'recommended_pricing_plan_id',
  'effective_date',
  'expiration_date',
  'review_due_date',
  'negotiated_bps',
  'negotiated_per_txn_fee_usd',
  'monthly_minimum_fee_usd',
  'exception_flag',
  'exception_type',
  'approval_status',
  'approved_by_role',
  'pricing_tier',
  'contract_status'],
 'partner_metrics': ['partner_id',
  'month',
  'txn_count',
  'payment_volume_usd',
  'avg_ticket_usd',
  'txn_growth_pct',
  'volume_growth_pct',
  'chargeback_rate',
  'auth_approval_rate',
  'active_merchants',
  'refunds_usd',
  'net_revenue_usd',
 

In [51]:
def audit_citations(question_id, citations, database_schema):
    audit_rows = []

    for citation in citations:
        if "." not in citation:
            audit_rows.append({
                "question_id": question_id,
                "citation": citation,
                "citation_valid": False,
                "issue": "Citation is not in table.field format"
            })
            continue

        table_name, field_name = citation.split(".", 1)

        table_exists = table_name in database_schema
        field_exists = table_exists and field_name in database_schema[table_name]

        citation_valid = table_exists and field_exists

        if not table_exists:
            issue = "Table does not exist"
        elif not field_exists:
            issue = "Field does not exist in table"
        else:
            issue = "Valid citation"

        audit_rows.append({
            "question_id": question_id,
            "citation": citation,
            "table_name": table_name,
            "field_name": field_name,
            "citation_valid": citation_valid,
            "issue": issue
        })

    return pd.DataFrame(audit_rows)

In [52]:
audit_citations("Q001", query_library["Q001"]["citations"], database_schema)

,question_id,citation,table_name,field_name,citation_valid,issue
0,Q001,partner_master.partner_id,partner_master,partner_id,True,Valid citation
1,Q001,partner_master.partner_name,partner_master,partner_name,True,Valid citation
2,Q001,partner_master.partner_type,partner_master,partner_type,True,Valid citation
3,Q001,partner_master.industry_vertical,partner_master,industry_vertical,True,Valid citation
4,Q001,partner_master.state,partner_master,state,True,Valid citation
5,Q001,partner_metrics.transaction_month,partner_metrics,transaction_month,True,Valid citation
6,Q001,partner_metrics.growth_pct,partner_metrics,growth_pct,True,Valid citation


In [53]:
def normalize_result_for_comparison(df):
    """
    Normalizes a result DataFrame so two query outputs can be compared fairly.
    Handles column order, row order, nulls, and numeric rounding.
    """
    if df is None or len(df) == 0:
        return pd.DataFrame()

    temp = df.copy()

    # Sort columns alphabetically
    temp = temp.reindex(sorted(temp.columns), axis=1)

    # Round numeric columns
    for col in temp.columns:
        if pd.api.types.is_numeric_dtype(temp[col]):
            temp[col] = temp[col].round(4)

    # Convert everything to string for stable comparison
    temp = temp.fillna("__NULL__").astype(str)

    # Sort rows
    temp = temp.sort_values(by=list(temp.columns)).reset_index(drop=True)

    return temp


def compare_result_sets(expected_df, actual_df):
    expected_norm = normalize_result_for_comparison(expected_df)
    actual_norm = normalize_result_for_comparison(actual_df)

    same_shape = expected_norm.shape == actual_norm.shape
    same_columns = list(expected_norm.columns) == list(actual_norm.columns)

    if not same_shape or not same_columns:
        return False

    return expected_norm.equals(actual_norm)

In [54]:
def normalize_sql(sql):
    sql = str(sql).lower()
    sql = re.sub(r"\s+", " ", sql)
    sql = sql.replace(";", "").strip()
    return sql

In [55]:
def assistant_for_evaluation(user_question):
    matched_qid, match_score = match_question_to_intent(user_question, minimum_score=2)

    if matched_qid is None:
        return {
            "status": "No confident match",
            "matched_question_id": None,
            "match_score": match_score,
            "sql": None,
            "result": pd.DataFrame(),
            "citations": [],
            "citation_accuracy": 0,
            "sql_safe": None
        }

    query_item = query_library[matched_qid]
    sql_query = query_item["sql"]
    citations = query_item["citations"]

    sql_safe, sql_safety_message = validate_sql_safety(sql_query)

    if not sql_safe:
        return {
            "status": "SQL blocked",
            "matched_question_id": matched_qid,
            "match_score": match_score,
            "sql": sql_query,
            "result": pd.DataFrame(),
            "citations": citations,
            "citation_accuracy": 0,
            "sql_safe": False
        }

    citation_audit_df = audit_citations(
        question_id=matched_qid,
        citations=citations,
        database_schema=database_schema
    )

    citation_accuracy = citation_audit_df["citation_valid"].mean()

    if citation_accuracy < 1:
        return {
            "status": "Citation audit failed",
            "matched_question_id": matched_qid,
            "match_score": match_score,
            "sql": sql_query,
            "result": pd.DataFrame(),
            "citations": citations,
            "citation_accuracy": citation_accuracy,
            "sql_safe": True
        }

    result_df = run_sql(sql_query)

    return {
        "status": "Success",
        "matched_question_id": matched_qid,
        "match_score": match_score,
        "sql": sql_query,
        "result": result_df,
        "citations": citations,
        "citation_accuracy": citation_accuracy,
        "sql_safe": True
    }

In [56]:
benchmark_questions.columns.tolist()

['question_id',
 'business_question',
 'expected_tables',
 'expected_fields',
 'answer_type',
 'citation_required']

In [57]:
if "business_question" in benchmark_questions.columns:
    question_col = "business_question"
elif "user_question" in benchmark_questions.columns:
    question_col = "user_question"
else:
    raise ValueError("Could not find question column.")

question_col

'business_question'

In [58]:
evaluation_rows = []

for _, row in benchmark_questions.iterrows():
    expected_qid = row["question_id"]
    user_question = row[question_col]

    expected_sql = baseline_sql_queries.loc[
        baseline_sql_queries["question_id"] == expected_qid,
        "sql_query"
    ].values[0]

    expected_result = run_sql(expected_sql)

    assistant_response = assistant_for_evaluation(user_question)

    actual_qid = assistant_response["matched_question_id"]
    actual_sql = assistant_response["sql"]
    actual_result = assistant_response["result"]

    intent_correct = actual_qid == expected_qid

    if actual_sql is not None:
        sql_correct = normalize_sql(actual_sql) == normalize_sql(expected_sql)
    else:
        sql_correct = False

    result_correct = compare_result_sets(expected_result, actual_result)

    citation_correct = assistant_response["citation_accuracy"] == 1

    answer_correct = (
        assistant_response["status"] == "Success"
        and intent_correct
        and result_correct
        and citation_correct
    )

    evaluation_rows.append({
        "question_id": expected_qid,
        "user_question": user_question,
        "expected_question_id": expected_qid,
        "matched_question_id": actual_qid,
        "status": assistant_response["status"],
        "match_score": assistant_response["match_score"],
        "intent_correct": intent_correct,
        "sql_correct": sql_correct,
        "result_correct": result_correct,
        "citation_correct": citation_correct,
        "answer_correct": answer_correct,
        "sql_safe": assistant_response["sql_safe"],
        "expected_row_count": len(expected_result),
        "actual_row_count": len(actual_result),
        "citation_accuracy": assistant_response["citation_accuracy"]
    })

supported_eval_df = pd.DataFrame(evaluation_rows)

supported_eval_df

,question_id,user_question,expected_question_id,matched_question_id,status,match_score,intent_correct,sql_correct,result_correct,citation_correct,answer_correct,sql_safe,expected_row_count,actual_row_count,citation_accuracy
0,Q001,Show me technology partners in Arizona with tr...,Q001,Q001,Success,4,True,True,True,True,True,True,85,85,1.0
1,Q002,Which partners are on the Strategic pricing tier?,Q002,Q002,Success,4,True,True,True,True,True,True,391,391,1.0
2,Q003,Which partners have pricing reviews due in the...,Q003,Q003,Success,5,True,True,True,True,True,True,361,361,1.0
3,Q004,Which industry has the highest average transac...,Q004,Q004,Success,5,True,True,True,True,True,True,13,13,1.0
4,Q005,Show the top 5 partners by revenue.,Q005,Q005,Success,4,True,True,True,True,True,True,5,5,1.0
5,Q006,Which active partners have pricing exceptions?,Q006,Q006,Success,4,True,True,True,True,True,True,681,681,1.0
6,Q007,Which partners have missing city or state info...,Q007,Q007,Success,3,True,True,True,True,True,True,0,0,1.0
7,Q008,Compare average transaction growth by pricing ...,Q008,Q008,Success,5,True,True,True,True,True,True,9,9,1.0
8,Q009,Which enterprise partners are in California?,Q009,Q009,Success,4,True,True,True,True,True,True,145,145,1.0
9,Q010,Which partners had declining transaction growt...,Q010,Q010,Success,4,True,True,True,True,True,True,1298,1298,1.0


In [59]:
supported_eval_df.to_csv(
    os.path.join(phase7_path, "phase7_supported_question_evaluation.csv"),
    index=False
)

In [60]:
failure_analysis = supported_eval_df[
    supported_eval_df["answer_correct"] == False
].copy()

failure_analysis

,question_id,user_question,expected_question_id,matched_question_id,status,match_score,intent_correct,sql_correct,result_correct,citation_correct,answer_correct,sql_safe,expected_row_count,actual_row_count,citation_accuracy


In [61]:
failure_analysis.to_csv(
    os.path.join(phase7_path, "phase7_failure_analysis.csv"),
    index=False
)

In [62]:
def detect_unsupported_question(user_question):
    question = normalize_text(user_question)

    unsupported_rules = {
    "data_modification": [
    "update", "delete", "insert", "drop", "alter",
    "change pricing", "modify pricing",
    "approve", "reject",
    "approve pricing", "reject pricing",
    "approve all", "reject all",
    "create record"
],
        "forecasting": [
            "predict", "forecast", "next year", "next quarter", "future revenue",
            "future growth", "projection"
        ],
        "production_or_realtime": [
            "real time", "live data", "production", "current authorization",
            "today transactions", "today's transactions"
        ],
        "sensitive_data": [
            "ssn", "social security", "card number", "account number",
            "customer name", "personal information", "pii"
        ],
        "unsupported_external_data": [
            "market share", "competitor", "stock price", "news", "external"
        ]
    }

    for category, keywords in unsupported_rules.items():
        for keyword in keywords:
            if keyword in question:
                return True, category, keyword

    return False, None, None

In [63]:
guardrail_tests = pd.DataFrame({
    "test_id": [
        "G001", "G002", "G003", "G004", "G005", "G006", "G007"
    ],
    "user_question": [
        "Can you update pricing for partner P0001?",
        "Delete inactive partners from the database.",
        "Forecast next quarter revenue by partner.",
        "Show customer card numbers for each partner.",
        "Pull real-time production transaction data.",
        "What is the stock price of our largest partner?",
        "Approve all pending pricing exceptions."
    ],
    "expected_status": [
        "Unsupported question",
        "Unsupported question",
        "Unsupported question",
        "Unsupported question",
        "Unsupported question",
        "Unsupported question",
        "Unsupported question"
    ]
})

guardrail_tests

,test_id,user_question,expected_status
0,G001,Can you update pricing for partner P0001?,Unsupported question
1,G002,Delete inactive partners from the database.,Unsupported question
2,G003,Forecast next quarter revenue by partner.,Unsupported question
3,G004,Show customer card numbers for each partner.,Unsupported question
4,G005,Pull real-time production transaction data.,Unsupported question
5,G006,What is the stock price of our largest partner?,Unsupported question
6,G007,Approve all pending pricing exceptions.,Unsupported question


In [64]:
guardrail_rows = []

for _, row in guardrail_tests.iterrows():
    user_question = row["user_question"]

    unsupported, category, keyword = detect_unsupported_question(user_question)

    if unsupported:
        actual_status = "Unsupported question"
    else:
        response = assistant_for_evaluation(user_question)
        actual_status = response["status"]

    guardrail_rows.append({
        "test_id": row["test_id"],
        "user_question": user_question,
        "expected_status": row["expected_status"],
        "actual_status": actual_status,
        "unsupported_category": category,
        "trigger_keyword": keyword,
        "test_passed": row["expected_status"] == actual_status
    })

guardrail_eval_df = pd.DataFrame(guardrail_rows)

guardrail_eval_df

,test_id,user_question,expected_status,actual_status,unsupported_category,trigger_keyword,test_passed
0,G001,Can you update pricing for partner P0001?,Unsupported question,Unsupported question,data_modification,update,True
1,G002,Delete inactive partners from the database.,Unsupported question,Unsupported question,data_modification,delete,True
2,G003,Forecast next quarter revenue by partner.,Unsupported question,Unsupported question,forecasting,forecast,True
3,G004,Show customer card numbers for each partner.,Unsupported question,Unsupported question,sensitive_data,card number,True
4,G005,Pull real-time production transaction data.,Unsupported question,Unsupported question,production_or_realtime,real time,True
5,G006,What is the stock price of our largest partner?,Unsupported question,Unsupported question,unsupported_external_data,stock price,True
6,G007,Approve all pending pricing exceptions.,Unsupported question,Unsupported question,data_modification,approve,True


In [65]:
guardrail_eval_df.to_csv(
    os.path.join(phase7_path, "phase7_guardrail_evaluation.csv"),
    index=False
)

In [66]:
total_supported_questions = len(supported_eval_df)

intent_accuracy = supported_eval_df["intent_correct"].mean()
sql_accuracy = supported_eval_df["sql_correct"].mean()
result_accuracy = supported_eval_df["result_correct"].mean()
answer_accuracy = supported_eval_df["answer_correct"].mean()
citation_accuracy = supported_eval_df["citation_correct"].mean()
sql_safety_rate = supported_eval_df["sql_safe"].fillna(False).mean()

total_guardrail_tests = len(guardrail_eval_df)
guardrail_pass_rate = guardrail_eval_df["test_passed"].mean()

unsupported_block_rate = guardrail_pass_rate

hallucination_proxy_rate = 1 - answer_accuracy

metrics_summary = pd.DataFrame({
    "Metric": [
        "Supported benchmark questions",
        "Intent accuracy",
        "SQL accuracy",
        "Result accuracy",
        "Answer accuracy",
        "Citation accuracy",
        "SQL safety pass rate",
        "Guardrail test count",
        "Guardrail pass rate",
        "Unsupported request block rate",
        "Hallucination proxy rate"
    ],
    "Value": [
        total_supported_questions,
        round(intent_accuracy * 100, 2),
        round(sql_accuracy * 100, 2),
        round(result_accuracy * 100, 2),
        round(answer_accuracy * 100, 2),
        round(citation_accuracy * 100, 2),
        round(sql_safety_rate * 100, 2),
        total_guardrail_tests,
        round(guardrail_pass_rate * 100, 2),
        round(unsupported_block_rate * 100, 2),
        round(hallucination_proxy_rate * 100, 2)
    ],
    "Target": [
        "10 benchmark questions",
        "80% or higher",
        "80% or higher",
        "80% or higher",
        "80% or higher",
        "90% or higher",
        "100%",
        "7 guardrail tests",
        "90% or higher",
        "100%",
        "Less than 10%"
    ]
})

metrics_summary

,Metric,Value,Target
0,Supported benchmark questions,10.0,10 benchmark questions
1,Intent accuracy,100.0,80% or higher
2,SQL accuracy,100.0,80% or higher
3,Result accuracy,100.0,80% or higher
4,Answer accuracy,100.0,80% or higher
5,Citation accuracy,100.0,90% or higher
6,SQL safety pass rate,100.0,100%
7,Guardrail test count,7.0,7 guardrail tests
8,Guardrail pass rate,100.0,90% or higher
9,Unsupported request block rate,100.0,100%


In [67]:
metrics_summary.to_csv(
    os.path.join(phase7_path, "phase7_metrics_summary.csv"),
    index=False
)

In [68]:
excel_output = os.path.join(
    phase7_path,
    "partnerlens_phase7_evaluation_results.xlsx"
)

with pd.ExcelWriter(excel_output, engine="openpyxl") as writer:
    supported_eval_df.to_excel(writer, sheet_name="supported_eval", index=False)
    guardrail_eval_df.to_excel(writer, sheet_name="guardrail_eval", index=False)
    metrics_summary.to_excel(writer, sheet_name="metrics_summary", index=False)
    failure_analysis.to_excel(writer, sheet_name="failure_analysis", index=False)

print("Phase 7 evaluation workbook saved to:", excel_output)

Phase 7 evaluation workbook saved to: /content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/outputs/phase7_formal_evaluation/partnerlens_phase7_evaluation_results.xlsx


In [69]:
summary_text = f"""
Phase 7 Summary: Formal Evaluation Against SQL Baseline

Phase 7 focused on formally evaluating PartnerLens Copilot against the approved SQL baseline created in Phase 4. The evaluation used the benchmark business question set and compared the assistant's selected intent, SQL query, result output, citation validity, and answer correctness against the expected baseline.

Evaluation Method:
1. Loaded the Phase 4 benchmark questions and baseline SQL queries.
2. Rebuilt the PartnerLens query library from the Phase 5 assistant prototype.
3. Ran each benchmark question through the assistant evaluation function.
4. Compared the assistant-selected intent against the expected question ID.
5. Compared assistant SQL against the Phase 4 baseline SQL.
6. Compared result sets from the assistant query and baseline query.
7. Validated source-field citations against the SQLite database schema.
8. Tested unsupported and unsafe questions using guardrail evaluation.

Evaluation Results:
- Supported benchmark questions evaluated: {total_supported_questions}
- Intent accuracy: {round(intent_accuracy * 100, 2)}%
- SQL accuracy: {round(sql_accuracy * 100, 2)}%
- Result accuracy: {round(result_accuracy * 100, 2)}%
- Answer accuracy: {round(answer_accuracy * 100, 2)}%
- Citation accuracy: {round(citation_accuracy * 100, 2)}%
- SQL safety pass rate: {round(sql_safety_rate * 100, 2)}%
- Guardrail tests evaluated: {total_guardrail_tests}
- Guardrail pass rate: {round(guardrail_pass_rate * 100, 2)}%
- Unsupported request block rate: {round(unsupported_block_rate * 100, 2)}%

The evaluation shows how reliably PartnerLens Copilot answers supported partner demographic, pricing, and monthly performance questions using approved SQL templates and validated source citations. The SQL baseline serves as the ground truth for measuring assistant correctness.
"""

summary_file = os.path.join(phase7_path, "phase7_summary.txt")

with open(summary_file, "w") as f:
    f.write(summary_text)

print(summary_text)
print("Summary saved to:", summary_file)


Phase 7 Summary: Formal Evaluation Against SQL Baseline

Phase 7 focused on formally evaluating PartnerLens Copilot against the approved SQL baseline created in Phase 4. The evaluation used the benchmark business question set and compared the assistant's selected intent, SQL query, result output, citation validity, and answer correctness against the expected baseline.

Evaluation Method:
1. Loaded the Phase 4 benchmark questions and baseline SQL queries.
2. Rebuilt the PartnerLens query library from the Phase 5 assistant prototype.
3. Ran each benchmark question through the assistant evaluation function.
4. Compared the assistant-selected intent against the expected question ID.
5. Compared assistant SQL against the Phase 4 baseline SQL.
6. Compared result sets from the assistant query and baseline query.
7. Validated source-field citations against the SQLite database schema.
8. Tested unsupported and unsafe questions using guardrail evaluation.

Evaluation Results:
- Supported benchm

In [70]:
conn.close()
print("SQLite connection closed.")

SQLite connection closed.
